# DIMMS and RINGSS Nightly Summary

Author(s): Bruno Quint
Last Update: 2025-10-24


In [ ]:
day_obs = 20251020

In [ ]:
# Clustering parameters (optimized for noisy data)
CLUSTERING_EPS = 0.1  # degrees - spatial tolerance for grouping observations
CLUSTERING_MIN_SAMPLES = 20  # minimum observations to form a cluster (increased from 5)

# Quality filtering thresholds
MIN_OBSERVATIONS = 50  # minimum observations to be considered a significant target
MIN_DURATION_MINUTES = 10  # OR minimum duration in minutes

# Shared target matching tolerances
SHARED_TARGET_RA_TOLERANCE = 0.15  # degrees
SHARED_TARGET_DECL_TOLERANCE = 0.15  # degrees

# Temporal overlap detection
SIMULTANEOUS_TIME_WINDOW_MINUTES = 5  # time window for "simultaneous" observations

# Vera Rubin Observatory latitude 
RUBIN_LATITUDE = -30.2446  # degrees
RUBIN_LONGITUDE = -70.7366  # degrees
SUN_ALTITUDE_LIMIT = -12.0  # degrees for twilight calculations

# Maximum elevation angle used to calculate tracking speed
maximum_elevation_angle = 85 # degrees

## Setup notebook

In [ ]:
import os    
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import astropy.units as u
    
from astropy.coordinates import EarthLocation, get_sun, AltAz
from astropy.time import Time
from IPython.display import display, FileLink, HTML
from sklearn.cluster import DBSCAN

from lsst.ts.xml.enums.DIMM import ScopeMotion

from lsst.summit.utils.efdUtils import (
    calcNextDay,
    getDayObsStartTime, 
    getDayObsEndTime, 
    getEfdData, 
    makeEfdClient
)

from dimm_single_night import foo

In [ ]:
foo()

In [ ]:
efd_client = makeEfdClient()

topics_and_columns = {
    "lsst.sal.DIMM.status": [
        "ra", "decl", "azimuth", "altitude", "motionState", "salIndex"
    ]
}

In [ ]:
def query_dimm_data(_day_obs: int, key: str = "lsst.sal.DIMM.status") -> pd.DataFrame:
    """
    Query and preprocess DIMM telemetry data for a given observing day.
    
    Retrieves DIMM status data from the EFD (Engineering Facilities Database),
    filters out invalid entries, and adds human-readable labels for motion states
    and hardware identifiers.
    
    Parameters:
    -----------
    _day_obs : int
        Observing day in YYYYMMDD format (e.g., 20251020)
    key : str, optional
        EFD topic key to query (default: "lsst.sal.DIMM.status")
        
    Returns:
    --------
    pd.DataFrame
        Processed DIMM data with columns including:
        - ra, decl: Right ascension and declination (degrees)
        - azimuth, altitude: Telescope pointing (degrees)
        - motionState: Numeric motion state code
        - motionStateName: Human-readable motion state (e.g., "Tracking", "Park")
        - salIndex: Hardware identifier (1 = Tower DIMM, 2 = Portable DIMM)
        - hardware: Human-readable hardware name
        - Other columns as defined in topics_and_columns[key]
        
    Notes:
    ------
    - Filters out rows with all NaN values
    - Removes entries where both RA and Decl are zero (invalid positions)
    - Only includes rows with valid (non-null) RA and Decl values
    - Requires pre-existing efd_client and topics_and_columns variables
    
    Example:
    --------
    >>> dimm_df = query_dimm_data(20251020)
    >>> print(f"Retrieved {len(dimm_df)} DIMM observations")
    """
    start_time = getDayObsStartTime(_day_obs)
    end_time = getDayObsEndTime(_day_obs)
    
    df = getEfdData(
        client=efd_client, 
        topic=key,
        columns=topics_and_columns[key],
        begin=start_time,
        end=end_time
    )
    
    # Our data has lots of NaN's. Let's drop them for now.
    df = df.dropna(how="all")
    # Clear out rows where ra/dec are both 0.
    df = df[df["ra"] != 0]
    df = df[df["decl"] != 0]
    # Map the motion state so it is human readable
    df["motionStateName"] = df["motionState"].map(lambda x: ScopeMotion(x).name)
    # Map salIndex to hardware names
    hardware_names = {1: 'Tower DIMM', 2: 'Portable DIMM'}
    df['hardware'] = df['salIndex'].map(hardware_names)
    # Filter valid data (non-null RA/DECL)
    df = df[(df['ra'].notna()) & (df['decl'].notna())].copy()
    return df


def get_max_tracking_speeds(latitude, elevation):
    """
    Calculate maximum azimuth and elevation tracking speeds for an alt-az telescope.
    
    Computes the theoretical maximum tracking rates needed to follow celestial
    objects due to Earth's rotation at a given observing location and elevation.
    
    Parameters:
    -----------
    latitude : float
        Observer's latitude in degrees (-90 to +90)
        Positive = North, Negative = South
    elevation : float
        Telescope elevation angle in degrees (0 to 90)
        
    Returns:
    --------
    tuple : (max_az_rate, max_el_rate, max_total_speed)
        max_az_rate : float
            Maximum azimuth tracking rate in deg/s
        max_el_rate : float
            Maximum elevation tracking rate in deg/s
        max_total_speed : float
            Maximum total angular speed in deg/s
            
    Notes:
    ------
    Maximum rates occur for an equatorial target (dec=0°) crossing the meridian.
    - Azimuth rate: ω / cos(elevation) - increases dramatically near zenith
    - Elevation rate: ω × |sin(latitude)| - relatively constant for a site
    - Earth's sidereal rotation: ω = 0.00417807462 deg/s (15.041 arcsec/s)
    
    Examples:
    --------
    >>> # Rubin Observatory at 45° elevation
    >>> az, el, total = get_max_tracking_speeds(-30.24, 45)
    >>> print(f"Az: {az*3600:.1f} arcsec/s, Total: {total*3600:.1f} arcsec/s")
    Az: 21.3 arcsec/s, Total: 22.6 arcsec/s
    
    >>> # Near zenith (85° elevation)
    >>> az, el, total = get_max_tracking_speeds(-30.24, 85)
    >>> print(f"Az: {az*3600:.1f} arcsec/s")
    Az: 172.6 arcsec/s
    """
    import numpy as np
    
    OMEGA = 0.00417807462  # Earth's rotation in deg/s
    
    lat_rad = np.deg2rad(latitude)
    elev_rad = np.deg2rad(elevation)
    
    max_az = OMEGA / np.cos(elev_rad)
    max_el = OMEGA * np.abs(np.sin(lat_rad))
    max_total = np.sqrt(max_az**2 + max_el**2)
    
    return max_az, max_el, max_total


def calculate_az_el_speeds(df):
    """
    Calculate azimuth and elevation tracking speeds from DIMM data.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'azimuth', 'altitude' columns and datetime index
        
    Returns:
    --------
    pd.DataFrame
        Input DataFrame with added columns:
        - 'az_speed': azimuth rate (deg/s)
        - 'el_speed': elevation rate (deg/s)
        - 'angular_speed': total angular speed (deg/s)
    """
    import numpy as np
    
    df = df.copy()
    
    # Initialize
    df['az_speed'] = np.nan
    df['el_speed'] = np.nan
    df['angular_speed'] = np.nan
    
    # Calculate separately for each hardware
    for sal_idx in df['salIndex'].unique():
        mask = df['salIndex'] == sal_idx
        data = df[mask].sort_index()
        
        # Get consecutive differences
        az_diff = data['azimuth'].diff().values
        alt_diff = data['altitude'].diff().values
        time_diff = data.index.to_series().diff().dt.total_seconds().values
        
        # Avoid division by zero
        time_diff[time_diff == 0] = np.nan
        
        # Calculate speeds (deg/s)
        az_speed = np.abs(az_diff) / time_diff
        el_speed = np.abs(alt_diff) / time_diff
        total_speed = np.sqrt(az_speed**2 + el_speed**2)
        
        # Assign back
        df.loc[mask, 'az_speed'] = az_speed
        df.loc[mask, 'el_speed'] = el_speed
        df.loc[mask, 'angular_speed'] = total_speed
    
    return df


def get_astronomical_twilight_times(day_obs, latitude=RUBIN_LATITUDE, longitude=RUBIN_LONGITUDE):
    """
    Calculate astronomical twilight times for a given day at Rubin Observatory.
    
    Parameters:
    -----------
    day_obs : int
        Observation day in YYYYMMDD format
    latitude : float
        Observatory latitude in degrees (default: Rubin Observatory)
    longitude : float  
        Observatory longitude in degrees (default: Rubin Observatory)
        
    Returns:
    --------
    tuple
        (evening_twilight_start_utc, morning_twilight_end_utc) as pandas Timestamps (timezone-aware UTC)
    """

    # Rubin Observatory location
    rubin_location = EarthLocation(lat=latitude*u.deg, lon=longitude*u.deg, height=2663*u.m)
    
    # Convert day_obs to date
    time_start = getDayObsStartTime(day_obs)
    time_end = getDayObsEndTime(day_obs)
    
    # Create time array for the day (every 10 minutes to find twilight transitions)
    # Use explicit start/end with 10-minute cadence
    step_sec = 10 * 60  # 10 minutes in seconds
    total_seconds = (time_end - time_start).to(u.s).value
    if total_seconds <= 0:
        times = Time([time_start])
    else:
        steps = np.arange(0, total_seconds + step_sec, step_sec)
        times = time_start + steps * u.s
    
    # Calculate sun altitude throughout the day
    sun = get_sun(times)
    sun_altaz = sun.transform_to(AltAz(obstime=times, location=rubin_location))
    sun_altitude = sun_altaz.alt.degree
    
    # Define astronomical twilight limit
    astronomical_limit = SUN_ALTITUDE_LIMIT
    
    # Find twilight transitions
    below_limit = sun_altitude < astronomical_limit
    transitions = np.diff(below_limit.astype(int))
    
    evening_start = None
    morning_end = None
    
    # Find evening twilight start (sun goes below -18°)
    evening_transitions = np.where(transitions == 1)[0]  # False to True
    if len(evening_transitions) > 0:
        idx = evening_transitions[0]
        evening_start = times[idx + 1].to_datetime()
        
    # Find morning twilight end (sun comes above -18°) 
    morning_transitions = np.where(transitions == -1)[0]  # True to False
    if len(morning_transitions) > 0:
        idx = morning_transitions[-1]  # Take the last transition (morning)
        morning_end = times[idx + 1].to_datetime()
    
    # Convert to pandas timestamps for consistency with your data
    # Ensure the timestamps are timezone-aware (UTC) so comparisons with dimm_df.index (tz-aware) succeed
    evening_start_pd = pd.to_datetime(evening_start, utc=True) if evening_start is not None else None
    morning_end_pd = pd.to_datetime(morning_end, utc=True) if morning_end is not None else None
    
    return evening_start_pd, morning_end_pd


def cluster_targets(data, hardware_name, eps=CLUSTERING_EPS, min_samples=CLUSTERING_MIN_SAMPLES):
    """
    Cluster targets using DBSCAN algorithm with robust parameters
    
    Parameters:
    -----------
    data : DataFrame
        Tracking data for a single hardware
    hardware_name : str
        Name of the hardware (for display)
    eps : float
        Maximum distance between points in same cluster (degrees)
    min_samples : int
        Minimum points to form a cluster
    
    Returns:
    --------
    DataFrame with target_id column added, dict with statistics
    """
    # Prepare coordinates
    coords = data[['ra', 'decl']].values
    
    # Perform clustering
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    
    # Add cluster labels
    data_clustered = data.copy()
    data_clustered['target_id'] = clustering.labels_
    
    # Initial statistics
    n_clusters_initial = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
    n_noise_initial = sum(clustering.labels_ == -1)
    
    # Calculate cluster statistics for filtering
    valid_data = data_clustered[data_clustered['target_id'] >= 0]
    
    if len(valid_data) > 0:
        cluster_stats = valid_data.groupby('target_id').agg({
            'ra': 'count'
        })
        cluster_stats.columns = ['count']
        
        # Calculate durations
        time_info = valid_data.groupby('target_id').apply(
            lambda x: (x.index.max() - x.index.min()).total_seconds() / 60,
            include_groups=False
        )
        cluster_stats['duration_min'] = time_info
        
        # Apply quality filters
        significant_mask = (
            (cluster_stats['count'] >= MIN_OBSERVATIONS) | 
            (cluster_stats['duration_min'] >= MIN_DURATION_MINUTES)
        )
        significant_cluster_ids = cluster_stats[significant_mask].index.tolist()
        
        # Filter the data to keep only significant clusters
        data_clustered.loc[
            (data_clustered['target_id'] >= 0) & 
            (~data_clustered['target_id'].isin(significant_cluster_ids)),
            'target_id'
        ] = -1
        
        # Renumber clusters sequentially starting from 0
        old_to_new = {old_id: new_id for new_id, old_id in enumerate(sorted(significant_cluster_ids))}
        data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'] = \
            data_clustered.loc[data_clustered['target_id'] >= 0, 'target_id'].map(old_to_new)
        
        n_clusters_final = len(significant_cluster_ids)
        n_noise_final = sum(data_clustered['target_id'] == -1)
        n_filtered = n_clusters_initial - n_clusters_final
        
        print(f"{hardware_name}: {len(data):,} records → {n_clusters_initial} initial clusters → {n_clusters_final} final clusters ({n_filtered} filtered, {n_noise_final} noise points)")
    else:
        n_clusters_final = 0
        n_filtered = 0
        n_noise_final = n_noise_initial
        print(f"{hardware_name}: {len(data):,} records → {n_clusters_initial} initial clusters → 0 final clusters ({n_noise_initial} noise points)")
    
    stats = {
        'initial_clusters': n_clusters_initial,
        'final_clusters': n_clusters_final,
        'filtered_clusters': n_filtered,
        'noise_points': n_noise_final
    }
    
    return data_clustered, stats


def find_shared_targets(_tower_df, _portable_df, 
                       ra_tolerance=SHARED_TARGET_RA_TOLERANCE, 
                       decl_tolerance=SHARED_TARGET_DECL_TOLERANCE):
    """
    Find targets observed by both DIMMs based on RA/DECL matching and include timing information.
    
    Parameters:
    -----------
    _tower_df : DataFrame
        Clustered data from Tower DIMM
    _portable_df : DataFrame
        Clustered data from Portable DIMM
    ra_tolerance : float
        Maximum RA difference to consider same target (degrees)
    decl_tolerance : float
        Maximum DECL difference to consider same target (degrees)
    
    Returns:
    --------
    DataFrame with matched targets including start/end timestamps
    """
    # Get target summaries for each hardware (only valid clusters)
    tower_targets = _tower_df[_tower_df['target_id'] >= 0].groupby('target_id').agg({
        'ra': 'mean',
        'decl': 'mean',
    }).reset_index()
    tower_targets.columns = ['tower_target_id', 'tower_ra', 'tower_decl']
    
    portable_targets = _portable_df[_portable_df['target_id'] >= 0].groupby('target_id').agg({
        'ra': 'mean',
        'decl': 'mean',
    }).reset_index()
    portable_targets.columns = ['portable_target_id', 'portable_ra', 'portable_decl']
    
    # Find matches
    matches = []
    
    for _, tower_row in tower_targets.iterrows():
        for _, portable_row in portable_targets.iterrows():
            ra_diff = abs(tower_row['tower_ra'] - portable_row['portable_ra'])
            decl_diff = abs(tower_row['tower_decl'] - portable_row['portable_decl'])
            
            # Handle RA wraparound at 0/360 degrees
            if ra_diff > 180:
                ra_diff = 360 - ra_diff
            
            if ra_diff <= ra_tolerance and decl_diff <= decl_tolerance:
                # Get timing information for this shared target
                tower_target_data = _tower_df[_tower_df['target_id'] == tower_row['tower_target_id']]
                portable_target_data = _portable_df[_portable_df['target_id'] == portable_row['portable_target_id']]
                
                # Get start and end times for each DIMM
                tower_start = tower_target_data.index.min()
                tower_end = tower_target_data.index.max()
                portable_start = portable_target_data.index.min()
                portable_end = portable_target_data.index.max()
                
                # Calculate overlap period (when both are tracking the same target)
                overlap_start = max(tower_start, portable_start)
                overlap_end = min(tower_end, portable_end)
                
                # Check if there's actual temporal overlap
                has_overlap = overlap_start <= overlap_end
                overlap_duration_min = (overlap_end - overlap_start).total_seconds() / 60 if has_overlap else 0
                
                matches.append({
                    'tower_target_id': tower_row['tower_target_id'],
                    'portable_target_id': portable_row['portable_target_id'],
                    'mean_ra': (tower_row['tower_ra'] + portable_row['portable_ra']) / 2,
                    'mean_decl': (tower_row['tower_decl'] + portable_row['portable_decl']) / 2,
                    'ra_diff': ra_diff,
                    'decl_diff': decl_diff,
                    'tower_ra': tower_row['tower_ra'],
                    'tower_decl': tower_row['tower_decl'],
                    'portable_ra': portable_row['portable_ra'],
                    'portable_decl': portable_row['portable_decl'],
                    # Timing information
                    'tower_start_time': tower_start,
                    'tower_end_time': tower_end,
                    'tower_duration_min': (tower_end - tower_start).total_seconds() / 60,
                    'tower_observations': len(tower_target_data),
                    'portable_start_time': portable_start,
                    'portable_end_time': portable_end,
                    'portable_duration_min': (portable_end - portable_start).total_seconds() / 60,
                    'portable_observations': len(portable_target_data),
                    # Overlap information
                    'has_temporal_overlap': has_overlap,
                    'overlap_start_time': overlap_start if has_overlap else None,
                    'overlap_end_time': overlap_end if has_overlap else None,
                    'overlap_duration_min': overlap_duration_min
                })
    
    result_df = pd.DataFrame(matches)
    print(f"Found {len(result_df)} shared targets between Tower DIMM ({len(tower_targets)} targets) and Portable DIMM ({len(portable_targets)} targets)")
    return result_df


def show_dimm_tracking_data(
    _dimm_data:pd.DataFrame, 
    _shared_targets:pd.DataFrame, 
    _max_az_speed:float, 
    _max_el_speed:float,
    _day_obs:int
) -> plt.Figure:

    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

    # Radial azimuth/elevation tower dimm - unfiltered data
    tower_df = _dimm_data[_dimm_data['salIndex'] == 1]
    portable_df = _dimm_data[_dimm_data['salIndex'] == 2]

    t_speed_mask = (tower_df["az_speed"] <= _max_az_speed) & (tower_df["el_speed"] <= _max_el_speed)
    p_speed_mask = (portable_df["az_speed"] <= _max_az_speed) & (portable_df["el_speed"] <= _max_el_speed)

    ax0.plot(tower_df.index, tower_df["azimuth"], 'C0-', alpha=0.2)
    ax0.plot(portable_df.index, portable_df["azimuth"], 'C1-', alpha=0.2)

    ax1.plot(tower_df.index, tower_df["altitude"], 'C0-', alpha=0.2)
    ax1.plot(portable_df.index, portable_df["altitude"], 'C1-', alpha=0.2)
    ax0.plot(tower_df[t_speed_mask].index, tower_df["azimuth"][t_speed_mask], 'C0o', alpha=0.5, label='Tower DIMM')
    ax0.plot(portable_df[p_speed_mask].index, portable_df["azimuth"][p_speed_mask], 'C1o', alpha=0.5, label='Portable DIMM')
    ax0.legend()

    ax1.plot(tower_df[t_speed_mask].index, tower_df["altitude"][t_speed_mask], 'C0o', alpha=0.5)
    ax1.plot(portable_df[p_speed_mask].index, portable_df["altitude"][p_speed_mask], 'C1o', alpha=0.5)

    ax0.grid(":", alpha=0.5)
    ax0.set_ylabel("Azimuth [deg]")
    ax1.set_ylabel("Elevation [deg]")
    ax1.set_xlabel("Time [UTC]")
    ax1.grid(":", alpha=0.5)

    date_format = mdates.DateFormatter('%H:%M')
    ax1.xaxis.set_major_formatter(date_format)

    for _, row in _shared_targets.iterrows():
        
        if not row.get('has_temporal_overlap', False):
            continue
        
        start = row.get('overlap_start_time') or row.get('overlap_start')
        end = row.get('overlap_end_time') or row.get('overlap_end')
        if pd.isna(start) or pd.isna(end):
            continue
        
        # draw translucent vertical span on both axes
        ax0.axvspan(start, end, color='gold', alpha=0.15)
        ax1.axvspan(start, end, color='gold', alpha=0.15)
        
        # label the span with the matched target ids
        try:
            label = f"T{int(row['tower_target_id'])}/P{int(row['portable_target_id'])}"
        except Exception:
            label = ""
        
        if label:
            mid = start + (end - start) / 2
            y_top = ax0.get_ylim()[1]
            y_range = y_top - ax0.get_ylim()[0]
            y = y_top - 0.03 * y_range  # move label down 3% of axis height
            ax0.text(mid, y, label, ha='center', va='top', fontsize=8, color='k', alpha=0.7)


    fig.suptitle(f"Tower and Portable DIMM\nData filtered using tracking speed {_day_obs}")
    fig.autofmt_xdate()
    fig.tight_layout()
    # plt.show()
    return fig

# Query, clean and group the data

Let's start ensureing that we are comparing apples to apples.  
By the time we wrote this notebook, the `motionState` had unreliable data.  
The analysis will probably be easier when we have it.  

Let me start looping over a single `day_obs`.  
Then I will make a timeline plot for the two different DIMMs.  
After that, I used [sklearn.cluster.DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) to find out when we are tracking on sky.  
Not that this solution is temporary until we have `motionState` fixed with real values.  
Finally, we show the timeline again with tracking data being highlighted.  

In [ ]:
print(f"Start with day obs {day_obs}")
start_time = getDayObsStartTime(day_obs)
end_time = getDayObsEndTime(day_obs)

# # Query the data
dimm_df = query_dimm_data(day_obs)
print("Data frame size after initial cleaning:", len(dimm_df))

# Get maximum tracking speeds
max_az_speed, max_el_speed, max_total_speed = get_max_tracking_speeds(
    RUBIN_LATITUDE, maximum_elevation_angle)

# Calculate azimuth and elevation speeds
dimm_df = calculate_az_el_speeds(dimm_df)

speed_mask = (
    (dimm_df['az_speed'] <= max_az_speed) &
    (dimm_df['el_speed'] <= max_el_speed) &
    (dimm_df['angular_speed'] <= max_total_speed)
)
dimm_df = dimm_df[speed_mask].copy()
print("Data frame size after filtering based on tracking speed:", len(dimm_df))

# Ensure we have the proper data to run the analysis. 
# Stop if we don't.
if len(dimm_df[dimm_df['salIndex'] == 1]) == 0:
    raise ValueError(f"No valid Tower DIMM data for day obs: {day_obs}\n")

if len(dimm_df[dimm_df['salIndex'] == 1]) == 0:
    raise ValueError(f"No valid Portable DIMM data for day obs: {day_obs}\n")

# Cluster targets for each hardware
tower_df, tower_stats = cluster_targets(
    dimm_df[dimm_df['salIndex'] == 1],
    'Tower DIMM (salIndex=1)'
)

portable_df, portable_stats = cluster_targets(
    dimm_df[dimm_df['salIndex'] == 2],
    'Portable DIMM (salIndex=2)'
)

# Combine back into single dataframe with hardware-specific target IDs
tower_df['global_target_id'] = tower_df['target_id'].apply(
    lambda x: f"T{x}" if x >= 0 else "T-1"
)
portable_df['global_target_id'] = portable_df['target_id'].apply(
    lambda x: f"P{x}" if x >= 0 else "P-1"
)

dimm_df = pd.concat([tower_df, portable_df])
dimm_df = dimm_df.sort_index()


# Identify shared targets
shared_targets = find_shared_targets(tower_df, portable_df)

# Plot the data
fig = show_dimm_tracking_data(
    dimm_df, 
    shared_targets, 
    max_az_speed, 
    max_el_speed, 
    day_obs
)

# Save the data 
os.makedirs("data", exist_ok=True)
dimm_df.to_csv(f"data/dimm_data_{day_obs}.csv")
shared_targets.to_csv(f"data/shared_targets_{day_obs}.csv")
fig.savefig(f"data/dimm_data_{day_obs}.png")

# Create a link to the files produced above    
display(
    HTML(
        f"""
        <ul>
        <li><a href="data/dimm_data_{day_obs}.csv" target="_blank">DIMM Data CSV for {day_obs}</a><br></li>
        <li><a href="data/shared_targets_{day_obs}.csv" target="_blank">Shared Targets CSV for {day_obs}</a><br></li>
        <li><a href="data/dimm_data_{day_obs}.png" target="_blank">DIMM Data PNG for {day_obs}</a><br></li>
        </ul>
        """
    )
)

# Increment day_obs
print(f"Done with day obs: {day_obs}\n", 79*"-", "\n")